# Справка

1. Если __"Ваш номер заказа"__ == *NaN* -> __"Статус товара"__ == *"Заказ отменен до обработки"*

In [2]:
import pandas as pd
import re

In [3]:
file = 'united_orders_21570113_01-01-2025_15-01-2025.xlsx'

In [4]:
pd.ExcelFile(file).sheet_names

['Сводка',
 'Транзакции по заказам и товарам',
 'Услуги и маржа по заказам',
 'Лист1',
 'Заказы с умн.ценообразованием',
 'Сводка по Статусам',
 'Сводка по SKU']

In [4]:
# Получим даты отчета, чтобы отфильтровать артефакты
summary = pd.read_excel(file,
                   sheet_name='Сводка',
                   nrows=1,
                   header=None)
dates = re.findall(r"\d{2}\.\d{2}\.\d{4}", str(summary.iloc[0].values[0]))
dates_dt = pd.to_datetime(dates, format='%d.%m.%Y')


### Собираем данные с листа с услугами и маржой

In [5]:
financials = pd.read_excel(file, sheet_name='Услуги и маржа по заказам', header=6)
fin_cols = [
    'Номер заказа'
    , 'Цена продажи, ₽' # to float64
    , 'Все услуги Маркета за заказы, ₽'
    , 'Доход за вычетом услуг Маркета, ₽'
    , 'Статус платежа покупателя'
    , 'Размещение товаров на витрине, ₽'
    , 'Буст продаж, ₽'
    , 'Доставка покупателю, ₽'
    , 'Приём платежа покупателя, ₽'
    , 'Перевод платежа покупателя, ₽'
    , 'Обработка заказа, ₽'
]
financials = financials[fin_cols].fillna(0.0)

,Номер заказа,"Цена продажи, ₽","Все услуги Маркета за заказы, ₽","Доход за вычетом услуг Маркета, ₽",Статус платежа покупателя,"Размещение товаров на витрине, ₽","Буст продаж, ₽","Доставка покупателю, ₽","Приём платежа покупателя, ₽","Перевод платежа покупателя, ₽","Обработка заказа, ₽"
3828,38274834240,376,69.90,306.10,Переведён,1.0,37.04,16.92,0.12,4.82,10.0
464,40248584769,399,0.00,399.00,Будет переведён,0.0,0.00,0.00,0.00,0.00,0.0
1591,40262161536,415,0.00,415.00,Будет переведён,0.0,0.00,0.00,0.00,0.00,0.0
3572,40202530497,556,0.00,556.00,Будет переведён,0.0,0.00,0.00,0.00,0.00,0.0
1197,39568230146,550,41.06,508.94,Переведён,1.0,0.00,24.75,0.12,5.19,10.0


### Собираем данные с листа с транзакциями

In [6]:
transactions = pd.read_excel(file,
                   sheet_name='Транзакции по заказам и товарам',
                   header=8,
                   parse_dates=['Дата оформления'],
                   date_format='%d.%m.%Y')

In [7]:
for_stat = [
          'Номер заказа'
        # , 'Ваш номер заказа'
        , 'Дата оформления'
        # , 'Тип заказа'
        , 'Ваш SKU'
        , 'Название товара'
        , 'Количество'
        , 'Статус товара'
        # , 'Ваша цена, ₽'
        , 'Цена продажи, ₽'
        # , 'Передано в доставку'
       # 'Ваш порог снижения цены для участия в софинансировании на момент оформления заказа, ₽',
       # 'Ваша скидка, если товар участвовал в софинансировании, ₽',
       # 'Скидка маркетплейса (за шт.)', 'Unnamed: 20',
       # 'Оплата бонусами СберСпасибо (за шт.), ₽',
       # 'Оплата баллами Яндекс Плюса (за шт.), ₽',
       # 'Статус изменен', 'Способ оплаты', 'Склад отгрузки', 'Дата отгрузки',
       # 'Регион доставки', 'Сумма платежа', 'Номер платежного поручения',
       # 'Дата платежного поручения', 'Идентификатор платежа',
       # 'Дата реестра платежей', 'Сумма баллов', 'Статус (справочно)',
       # 'Сумма платежа.1', 'Номер платежного поручения.1',
       # 'Дата платежного поручения.1', 'Идентификатор платежа.1',
       # 'Дата реестра платежей.1', 'Сумма платежа.2',
       # 'Номер платежного поручения.2', 'Дата платежного поручения.2',
       # 'Идентификатор платежа.2', 'Дата реестра платежей.2', 'Сумма платежа.3',
       # 'Номер платежного поручения.3', 'Дата платежного поручения.3',
       # 'Идентификатор платежа.3', 'Дата реестра платежей.3', 'Сумма возврата',
       # 'Номер платежного поручения.4', 'Дата платежного поручения.4',
       # 'Идентификатор платежа.4', 'Дата реестра платежей.4',
       # 'Сумма возврата.1', 'Номер платежного поручения.5',
       # 'Дата платежного поручения.5', 'Идентификатор платежа.5',
       # 'Дата реестра платежей.5', 'Сумма возврата.2',
       # 'Номер платежного поручения.6', 'Дата платежного поручения.6',
       # 'Идентификатор платежа.6', 'Дата реестра платежей.6',
       # 'Сумма возврата.3', 'Номер платежного поручения.7',
       # 'Дата платежного поручения.7', 'Идентификатор платежа.7',
       # 'Дата реестра платежей.7', 'Удержанная сумма',
       # 'Номер платежного поручения.8', 'Дата платежного поручения.8',
       # 'Идентификатор платежа.8', 'Дата реестра платежей.8'
]

In [8]:
# фильтрация выбросов и артефактов
transactions = transactions[for_stat].dropna(subset=['Номер заказа']) # заказ без номера

# даты меньше отчетной на 60 дней или выше окончания отчета
min_date_condition = transactions['Дата оформления'] >= (dates_dt.min() - pd.Timedelta(days=60))
max_date_condition = transactions['Дата оформления'] <= dates_dt.max()
correct_period = min_date_condition & max_date_condition
transactions = transactions[correct_period]

# Преобразовать в целое число
to_int = ['Номер заказа'
          # , 'Ваш номер заказа'
          , 'Количество']

transactions[to_int] = transactions[to_int].apply(pd.to_numeric, errors='coerce').astype('Int64')

# Переименуем колонку "Цена продажи, ₽" для Транзакций, так как здесь только 1 товар, а не весь чек
transactions = transactions.rename(columns={'Цена продажи, ₽': 'Цена продажи позиции в чеке, ₽'})

In [9]:
# Добавим колонку "Стоимость товаров, ₽" = "Количество" * "Цена продажи, ₽"
transactions['Стоимость товаров, ₽'] = transactions['Количество'].mul(transactions['Цена продажи позиции в чеке, ₽'])

In [10]:
# Соединим транзации и расходы на услуги Маркета

transactions = transactions.merge(financials, on='Номер заказа', how='left')

fin_cols.remove('Номер заказа')
fin_cols.remove('Статус платежа покупателя')
fin_cols.remove('Цена продажи, ₽')

# 2. Коэффициент от чека (доля стоимости товаров от продажной цены)
transactions['Коэффициент от чека'] = transactions['Стоимость товаров, ₽'] / transactions['Цена продажи, ₽']

In [11]:
# 3. Пропорциональная часть услуг
for col in fin_cols:
    transactions[col] = (transactions[col] * transactions['Коэффициент от чека']).round(2)
transactions = transactions.drop(labels='Коэффициент от чека', axis=1)

In [12]:
result = transactions.groupby(['Статус товара', 'Ваш SKU']).agg(
    Количество_итого=('Количество', 'sum')
    ,Доход_итого=('Доход за вычетом услуг Маркета, ₽', 'sum')
    ,Услуги_итого=('Все услуги Маркета за заказы, ₽', 'sum')
    ,Max_cost=('Цена продажи позиции в чеке, ₽', 'max')
    ,Min_cost=('Цена продажи позиции в чеке, ₽', 'min')
    ,Mean_cost=('Цена продажи позиции в чеке, ₽', 'mean')
    ,Median_cost=('Цена продажи позиции в чеке, ₽', 'median')
    ,Quantile_cost_90=('Цена продажи позиции в чеке, ₽', lambda x: x.quantile(0.9))
).rename(columns={
     'Количество_итого': 'Итого: Количество, шт.'
    ,'Доход_итого': 'Доход за вычетом услуг Маркета, ₽'
    ,'Услуги_итого': 'Все услуги Маркета за заказы, ₽'
    ,'Max_cost': 'Максимальная цена продажи за период, ₽'
    ,'Min_cost': 'Минимальная цена продажи за период, ₽'
    ,'Mean_cost': 'Средняя цена продажи,  ₽'
    ,'Median_cost': 'Медианная цена продажи, ₽'
    ,'Quantile_cost_90': '90% цен ниже этого значения (квантиль), ₽' #
}).round(2)

result = result.reset_index().sort_values(by=['Статус товара', 'Доход за вычетом услуг Маркета, ₽'], ascending=[True, False])

In [13]:
summary_df = result.groupby('Статус товара')[['Доход за вычетом услуг Маркета, ₽', 'Все услуги Маркета за заказы, ₽']].sum()

In [14]:
result = result.merge(transactions.drop_duplicates(subset=['Ваш SKU'], keep='last')[['Ваш SKU', 'Название товара']], on='Ваш SKU', how='left')

In [15]:
with pd.ExcelWriter(file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    summary_df.to_excel(writer, sheet_name='Сводка по Статусам', index=True)
    result.to_excel(writer, sheet_name='Сводка по SKU', index=False)